## 1 Carga del dataset pre procesado
* Modulo scraper que entrega autos_autocosmos.csv sin nulos ni valores duplicados luego de realizar webscraping en autocosmos.com.ar 
* El modulo conversor carga el dataset entregado por el scraper, dolariza todos los precios mediante la funcion conversor que llama la API para obtener el valor del dolar al dia.

In [306]:
import sys
sys.path.append("..")

import pandas as pd
from config import RAW_DATA_PATH
from conversor import procesar_dataset 

df = procesar_dataset()
df.info()

Dólar blue a la fecha: 1550
✅ Archivo CSV dolarizado guardado en G:\Mi unidad\Price_pred\data\processed\autos_dolarizado.csv
<class 'pandas.DataFrame'>
RangeIndex: 4714 entries, 0 to 4713
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   marca       4714 non-null   str    
 1   modelo      4714 non-null   str    
 2   anio        4714 non-null   int64  
 3   km          4714 non-null   int64  
 4   ciudad      4714 non-null   str    
 5   provincia   4714 non-null   str    
 6   precio_usd  4714 non-null   float64
dtypes: float64(1), int64(2), str(4)
memory usage: 257.9 KB


## 2 Diagnóstico inicial 
### Variables cuantitativas, nulos, duplicados, rango de valores plausibles:

A verificar posibles errores y otliers:

* ¿Existen registros con 'km' = 0, pero años que no cohincide? 
* ¿Existen registros con 'anio' muy antiguo, de que tipo de vehiculo se trata?

In [307]:
df.head()

,marca,modelo,anio,km,ciudad,provincia,precio_usd
0,Chevrolet,Onix,2018,73000,Hurlingham,Buenos Aires (A.M.B.A.),6451.612903
1,Chevrolet,Cruze 5,2023,102000,Saavedra,Ciudad Autónoma Buenos Aires,16129.032258
2,Citroën,Berlingo Multispace,2017,84000,Flores,Ciudad Autónoma Buenos Aires,10903.225806
3,Renault,Kangoo,2014,84004,Flores,Ciudad Autónoma Buenos Aires,8322.580645
4,Volkswagen,Gol Trend,2020,20000,Flores,Ciudad Autónoma Buenos Aires,1290.321935


In [308]:
df['anio'].max() # existe el año 2027

df = df[df['anio'] != 2027]

# elimino los registros donde el valor es un error

### Histograma de valores en kilometraje 'km'

Cómo esta compuesto este campo? que valor predominan con respecto a la antiguedad/uso de los vehiculo que se ofrecen.

Verifiqué los registros con valores elevados o atípicos (mucho km y poca antiguedad o viseversa), para clasificar el caso, o verificar y corregir errores de carga.

In [309]:
import plotly.express as px

hist = px.histogram(df,
                    x = "km",
                    nbins=100,
                    title="Distribucion de kilometrajes",
                    template = 'plotly_dark',
                    height = 400,
                    width = 600).show()

### Verificacion de casos con bajo km, se verifica si es posible o realista en relacion al valor de 'anio', y km=100 (caso de placeholder) 


In [310]:
km_100 = df[df['km'] == 100].index
df = df.drop(km_100)


Verificado que km = 1 y 0, tienen un precio acorde a un auto 0km. Los conservo porque son parte del mercado actual

FIAT Cronos, FIAT Pulse, FIAT Argo, FIAT Fastback, y Chevrolet Onix/Tracker. Acá el precio está objetivamente mal.
km=21 repetido idéntico en 6 filas de Cronos
km=300 repetido en Onix/Tracker/Onix/Cronos/Tracker
un valor fijo reapareciendo en modelos y marcas distintas, otra vez la firma de un valor de fallback/cuota mal 
capturado por el scraper en vez del precio y/o km real

In [311]:
km_300 = df[df['km'] == 300]
km_21 = df[df['km'] == 21]

# elimino los registros donde km =21, 300
df = df[~df['km'].isin([21, 300])]


In [312]:
km_bajos = df[(df['km'] < 1000) & (df['anio']< 2025)].index

df.drop(km_bajos, inplace = True) 

In [313]:
precios_altos = df[df['precio_usd']>800000].index
df.drop(precios_altos, inplace=True)

### Grafico de scatter - antiguedad del vehiculo y el kilometraje declarado

Busco verificar la coherencia en general, que los vehiculos con mayor antiguedad tengan un km mas alto, y viceversa, y en caso contrario, verificar la veracidad o tratar ese dato erroneo.

Se obeserva una nube de datos donde las relaciones parecen coherentes, y dos clusteres dispersos, donde por un lado se encuentran vehiculos antiguos con baja cantidad de kilometraje, candidatos a 'Coleccion' y vehiculos con pocos años de antiguedad pero muchos kilometros acumulados (se chequean los precios)

In [314]:
scatter = px.scatter(df,
                     x = 'anio',
                     y = 'km',
                     template = 'plotly_dark',
                     hover_data = ['modelo', 'precio_usd'],
                     ).show()

In [315]:
# deteccion y verificacion de outliers o posibles categorias
autos_antiguos_coleccion = df[(df['km'] < 3000) & (df['anio']< 2000)]

# categoria creada luego de verificar los datos 
df.loc[[1428, 1748, 3627, 3990, 4010], 'categoria'] = 'coleccion'

display(autos_antiguos_coleccion)


,marca,modelo,anio,km,ciudad,provincia,precio_usd
1428,Daihatsu,Cuore,1994,1000,Vistalba,Mendoza,50322.580645
1748,Ford,T,1929,1000,Las Heras,Mendoza,19500.000000
3627,FIAT,600,1962,1000,Campana,Buenos Aires,10000.000000
3990,Ford,Falcon,1980,1000,Coronel Brandsen,Buenos Aires,3096.774194
4010,Ika,Estanciera,1970,1100,La Falda,Córdoba,10967.741935


In [316]:
# agrupar categorias generales (coleccion, pickup, sedan etc)
import pandas as pd
modelos_unicos = df[['marca', 'modelo']].drop_duplicates().sort_values(['marca', 'modelo'])

print(modelos_unicos.shape)

modelos_unicos.to_csv(r"G:\Mi unidad\Price_pred\notebooks\modelos_para_categorizar.csv", index=False)

(421, 2)


In [317]:
# A partir de este csv clasifico por tipo de vehiculo para obtener mas informacion util para el modelo
# sin tener que lidiar con las 421 modelos que aportan mas ruido que datos utiles para este caso
# categorizacion con Claude
modelos_unicos

,marca,modelo
1097,Agrale,Marruá
1571,Alfa Romeo,MiTo
2615,Alfa Romeo,Stelvio
1027,Audi,A1
302,Audi,A1 Sportback
...,...,...
341,Volvo,V70
3497,Volvo,XC60
326,Zanella,Force Truck
3044,smart,Forfour


In [318]:
categorias = pd.read_csv(r"G:\Mi unidad\Price_pred\notebooks\modelos_categorizados.csv")
df = df.merge(categorias, on=["marca", "modelo"], how="left")
df.drop(columns = ['categoria','modelo'], inplace = True)

In [319]:
df.head()

,marca,anio,km,ciudad,provincia,precio_usd,carroceria
0,Chevrolet,2018,73000,Hurlingham,Buenos Aires (A.M.B.A.),6451.612903,Hatchback
1,Chevrolet,2023,102000,Saavedra,Ciudad Autónoma Buenos Aires,16129.032258,Hatchback
2,Citroën,2017,84000,Flores,Ciudad Autónoma Buenos Aires,10903.225806,Minivan
3,Renault,2014,84004,Flores,Ciudad Autónoma Buenos Aires,8322.580645,Furgón
4,Volkswagen,2020,20000,Flores,Ciudad Autónoma Buenos Aires,1290.321935,Hatchback


In [320]:
out = df[(df['precio_usd'] > 80000)]

out

,marca,anio,km,ciudad,provincia,precio_usd,carroceria
254,FIAT,2006,300000,La Falda,Córdoba,258064.516129,Sedán
668,Citroën,2023,40000,Colegiales,Ciudad Autónoma Buenos Aires,132258.064516,Hatchback
1694,Mercedes,2021,27000,Villa Urquiza,Ciudad Autónoma Buenos Aires,96900.000000,SUV
1700,Audi,2021,27000,Villa Urquiza,Ciudad Autónoma Buenos Aires,99000.000000,SUV
2438,Porsche,2011,74000,Guaymallén,Mendoza,95000.000000,Sedán
3009,Ford,2025,12000,Rosario,Santa Fe,110000.000000,Pickup
3138,Volkswagen,2024,10000,Puerto Madero,Ciudad Autónoma Buenos Aires,225806.451613,Pickup
3749,Honda,2009,170000,Wilde,Buenos Aires (A.M.B.A.),110000.000000,Hatchback
4503,Honda,2019,126000,Bahía Blanca,Buenos Aires,240000.000000,SUV


In [321]:
df = df.drop(index=out.index)

In [322]:
hist = px.histogram(df,
                    x = 'carroceria',
                    template = 'plotly_dark',
                    title = 'Conteo de carroceria'
                    )
hist.show()


In [323]:
mediana_precio_carr = df.groupby('carroceria')['precio_usd'].median()
mediana_precio_carr


carroceria
Cabriolet     10750.000000
Camión        18500.000000
Clásico        3685.483871
Coupé         23000.000000
Furgón        10000.000000
Hatchback      8508.064516
Minivan        8508.064516
Pickup        19032.258065
SUV           14838.709677
Sedán         10967.741935
Utilitario    54750.000000
Wagon          7096.774194
Name: precio_usd, dtype: float64

### Box plot de valores de vehiculos por año (antiguedad) 
Se marcan los 'suspected outliers' y se verifica que estos son modelos de autos cuyo valor atipico esta justificado (Audi Q8, Mercedes, Camaro, etc vehiculos conocidamente caros)

podemos observar los casos de coleccion con precios mas elevados en años sub 1990, observamos como la media de precio sube a medida que aumenta 'anio', vehiculos mas nuevos, lo que es esperable, los outliers estan verificados y son veraces.

In [324]:
plot = px.box(df,
              x = 'anio',
              y = 'precio_usd',
              points = 'suspectedoutliers',
              template = 'plotly_dark',
              hover_data= ['marca','carroceria'],
              title = 'Evoucion de precios por antiguedad'
              ).show()

In [325]:
# Outliers de precio — AMBOS extremos (bajo Y alto)
# outlier detectado en el grafico de 1.22M Fiar Chronos se lo considera error de carga

df['precio_usd'] = df['precio_usd'].round(2)

outliers_precio = df[(df['precio_usd'] < 1000) | (df['precio_usd'] > 120000)].index

#outliers_precio 

df = df.drop(outliers_precio)




In [326]:
df = df.drop(index= [2473, 1713,3051])

In [327]:
# verifico la veracidad de los registros con menos de 1000 km
error = df[df['km'] < 1000]
error.count()
error



,marca,anio,km,ciudad,provincia,precio_usd,carroceria
216,Peugeot,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,23548.39,SUV
231,Chery,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,29000.00,SUV
301,Citroën,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,17419.35,SUV
388,BAIC,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,34000.00,SUV
390,Chery,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,29500.00,SUV
446,BAIC,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,31000.00,SUV
549,Peugeot,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,31683.10,Furgón
567,Peugeot,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,40455.48,SUV
751,Peugeot,2026,1,Colegiales,Ciudad Autónoma Buenos Aires,22322.58,Hatchback
754,Volkswagen,2026,1,Posadas,Misiones,19258.06,Hatchback


In [328]:
df.drop(index=[4198, 4203, 4235, 4275, 4472], inplace=True)


### Histograma de precios, luego de tratar los autliers de km

In [329]:
hist = px.histogram(df,
                    x = "precio_usd",
                    nbins=100,
                    title="Distribucion de precios",
                    template = 'plotly_dark',
                    height = 400,
                    width = 600).show()

### Verifico km en relacion a precio_usd, de los vehiculos de año menor a 1980

Tomo la lista y verifico con Claude la veracidad de los registros

In [330]:
#limpio ✅
outliers_km = df[(df['anio'] < 1980)]
outliers_km

,marca,anio,km,ciudad,provincia,precio_usd,carroceria
459,Chevrolet,1960,6000,San Martin,Buenos Aires (A.M.B.A.),14000.00,Pickup
844,Jeep,1963,99998,Ferré,Buenos Aires,10000.00,SUV
1432,Chevrolet,1970,10547,Mocoretá,Corrientes,4064.52,Pickup
1727,Ford,1929,1000,Las Heras,Mendoza,19500.00,Clásico
1857,Ford,1974,59000,Juarez Celman,Córdoba,2774.19,Pickup
2145,FIAT,1963,250000,Villa Allende,Córdoba,1612.90,Clásico
2385,Chevrolet,1947,77000,Parana,Entre Ríos,30000.00,Sedán
2957,Peugeot,1978,3000,Palermo,Ciudad Autónoma Buenos Aires,4999.00,Clásico
2962,Chevrolet,1966,100000,Nuñez,Ciudad Autónoma Buenos Aires,11500.00,Pickup
3413,FIAT,1967,97000,Ensenada,Buenos Aires,1935.48,Clásico


Estadisticas de las variables numericas listas

In [331]:
df.describe()

,anio,km,precio_usd
count,4614.000000,4614.000000,4614.000000
mean,2016.073906,114600.014088,12998.580065
std,6.811669,79836.704641,8844.681146
min,1929.000000,1.000000,1000.000000
25%,2013.000000,60000.000000,7419.350000
50%,2017.000000,105000.000000,10645.160000
75%,2020.000000,153000.000000,16000.000000
max,2026.000000,955000.000000,76900.000000


## Analisis de variables cualitativas

### Cardinalidad: 

Cómo estan compuestas las columnas cualitativas? Resisten one hot encoding? (previamente a utilizar KNN Regressor, evalué utilizar Random Forest Rregressor)
En este caso 'marca' y 'provincia' podrian resisistilo y aportar informacion util al modelo y no solo ruido cuando existen demasiados registros con 0.

In [332]:
df.head(3)

,marca,anio,km,ciudad,provincia,precio_usd,carroceria
0,Chevrolet,2018,73000,Hurlingham,Buenos Aires (A.M.B.A.),6451.61,Hatchback
1,Chevrolet,2023,102000,Saavedra,Ciudad Autónoma Buenos Aires,16129.03,Hatchback
2,Citroën,2017,84000,Flores,Ciudad Autónoma Buenos Aires,10903.23,Minivan


* Cuantas filas tiene cada marca? 
* Cuales marcas aportan informacion y cuantas tienen tan pocos registros que no resultan significativas? 
* Cuantas marcas concentran el mayor porcentaje de registros? 
* cuales pueden ser agrupados como 'otro' ?

In [333]:
# frecuencias relativas acumuladas: 

df['marca'].value_counts(normalize=True).cumsum()

marca
Volkswagen    0.161032
Peugeot       0.313827
Ford          0.440182
Chevrolet     0.537278
Renault       0.634157
FIAT          0.724534
Toyota        0.800607
Citroën       0.847204
Nissan        0.875163
Honda         0.894885
Jeep          0.913524
Mercedes      0.926094
BMW           0.937365
Audi          0.948635
Hyundai       0.957304
Chery         0.965323
KIA           0.970741
Suzuki        0.974426
DS            0.977026
RAM           0.979627
Dodge         0.982011
Mitsubishi    0.983962
MINI          0.985696
Lifan         0.986996
BAIC          0.988080
Subaru        0.989163
Alfa Romeo    0.990247
Geely         0.991331
Volvo         0.992198
SEAT          0.993065
smart         0.993715
Foton         0.994365
Chrysler      0.995015
Daihatsu      0.995665
DFSK          0.996099
Zanella       0.996532
JAC           0.996966
MG            0.997399
Haval         0.997833
Iveco         0.998266
Porsche       0.998700
Agrale        0.998916
Isuzu         0.999133
SWM  

In [334]:
'''
import numpy as np
umbral = 0.95
frecuencias = df['marca'].value_counts(normalize=True)
top_marcas = frecuencias[frecuencias.cumsum() <= umbral].index

df['carroceria'] = np.where(df['categoria'] == 'coleccion',
                           df['categoria'],
                           df['marca'].where(df['marca'].isin(top_marcas), 'Otros')
                           )
'''

# esta lineas de codigo agrupaban todas las marcas con baja cantidad de registros dentro de 'otros', 
# el agrupamiento por marca unicamente no permitia que el modelo entienda la jerarquia entre vehiculos dentro
# de cada marca ej: Corolla o Hilux tienen precios distintos y el modelo solo veia Toyota.
# Posteriormente cree la clasificacion por carroceria para darle esta infomacion al modelo con una cantidad
# acotada de registros, en lugar de dejar la columna 'modelo' que aportaba  mas de 400 registros distintos
# de los cuales el modelo seleccionado no podia aprender 

"\nimport numpy as np\numbral = 0.95\nfrecuencias = df['marca'].value_counts(normalize=True)\ntop_marcas = frecuencias[frecuencias.cumsum() <= umbral].index\n\ndf['carroceria'] = np.where(df['categoria'] == 'coleccion',\n                           df['categoria'],\n                           df['marca'].where(df['marca'].isin(top_marcas), 'Otros')\n                           )\n"

In [335]:
# en lugar de marca, estas son las columnas que debo encodear
df['carroceria'].unique()

<StringArray>
[ 'Hatchback',    'Minivan',     'Furgón',      'Sedán',        'SUV',
     'Pickup',    'Clásico',     'Camión',  'Cabriolet',      'Wagon',
      'Coupé', 'Utilitario']
Length: 12, dtype: str

In [336]:
import plotly.express as px

scatter_precio_categoria = px.scatter(df,
                                      x = 'carroceria',
                                      y = 'precio_usd',
                                      color = 'anio',
                                      template = 'plotly_dark',
                                      title = 'Distribucion de precios por categoria: carroceria',
                                      hover_data= 'marca')
scatter_precio_categoria.show()

### Impacto de la ubicacion geografica sobre los precios 
Analizando los valores de 'precios_usd', se observa que la mediana para todas las provincias es estable (entre 0-20k), pero si existen mayor cantidad de valores elevados y outlers en provincias marcadas.

Expandire el analisis con ANOVA, para determinar si la ubicacion geografica del vehiculo aporta informacion al modelo predictivo.

In [337]:
orden = df.groupby('provincia')['precio_usd'].median().sort_values(ascending=False).index

box = px.box(df,
             x = 'provincia',
             y = 'precio_usd',
             category_orders={'provincia': list(orden)},
             template = 'plotly_dark',
             hover_data='marca',
             ).show()

Analizando los rangos de precios por ciudad dentro de cada provincia, vemos que este tampoco esta determinado por la ubicacion. 

In [338]:
prov_interes = ['Buenos Aires (A.M.B.A.)', 'Córdoba', 'Neuquén', 'Santa Cruz']

df_filtro = df[df['provincia'].isin(prov_interes)]

segmentacion_prov_ciud = (df[df['provincia'].isin(prov_interes)]
                          .groupby(['provincia','ciudad'])['precio_usd']
                          .median().sort_values(ascending=False))

orden_ciudades = segmentacion_prov_ciud.index.get_level_values('ciudad').tolist()

box = px.box(df_filtro,
             x = 'ciudad',
             y = 'precio_usd',
             color = 'provincia',
             category_orders={'ciudad': orden_ciudades},
             template = 'plotly_dark',
             hover_data='marca',
             ).show()


📊 ANOVA (Analysis of Variance)
Qué mide: compara las medias de una variable numérica entre distintos grupos categóricos.

La prueba ANOVA mostró diferencias estadísticamente significativas entre las medias de los grupos (F = 3.91, p < 0.001). Por lo tanto, se rechaza la hipótesis nula de igualdad de medias

In [339]:
from scipy import stats

# HO = el precio promedio es igual en todas las provincias? 

grupos = [df[df['provincia'] == p]['precio_usd'] for p in df['provincia'].unique()]
f_stat, p_val = stats.f_oneway(*grupos)

# Cociente entre "cuánto varían los promedios entre provincias" y 
# "cuánto varía el precio dentro de cada provincia
print("F-statistic:", f_stat)

# el precio promedio sí difiere significativamente entre provincias, no es casualidad.
print("p-value:", p_val)


F-statistic: 3.9104463599567834
p-value: 4.3525743602629964e-10


In [340]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df['precio_usd'],
                          groups=df['provincia'],
                          alpha=0.05)

print(tukey)


                           Multiple Comparison of Means - Tukey HSD, FWER=0.05                           
           group1                       group2             meandiff  p-adj     lower      upper    reject
---------------------------------------------------------------------------------------------------------
                Buenos Aires      Buenos Aires (A.M.B.A.)  -675.8321 0.9996  -2419.4939  1067.8298  False
                Buenos Aires                    Catamarca   524.2632    1.0 -15603.9883 16652.5148  False
                Buenos Aires                        Chaco -3281.1986 0.5056  -7666.9005  1104.5033  False
                Buenos Aires                       Chubut  -242.2894    1.0  -7749.8485  7265.2698  False
                Buenos Aires Ciudad Autónoma Buenos Aires -2501.5804    0.0  -4201.1502  -802.0105   True
                Buenos Aires                   Corrientes   607.0177    1.0  -4389.9579  5603.9933  False
                Buenos Aires                  

La localidad parece estar asociada al precio, pero eso no significa necesariamente que la localidad sea la causa de que el vehículo tenga ese precio

con el analisis ANOVA encontré diferencias significativas entre provincias, pero el Tukey mostró que esas diferencias están concentradas en unos pocos pares y no son generalizadas entre todas las provincias.

Eso me permite plantear la siguiente hipotesis:

Provincia → puede estar capturando diferencias de composición del mercado, pero no necesariamente un efecto propio de la provincia

La variable provincia puede actuar como una variable proxy de la composición del mercado. Las diferencias observadas en el precio promedio entre provincias podrían estar explicadas, al menos parcialmente, por las características de los vehículos disponibles en cada una, como marca, modelo, año, kilometraje o segmento, en lugar de representar un efecto intrínseco de la provincia.

Quitaré la columna de ciudad, conservo solamente provincia pero estará sujeta su conservacion a las pruebas de entrenamiento segun el aporte que haga o no en RMSE, MAE y R²

In [341]:
df= df.drop(columns='ciudad')

Reemplazo la columna de anio por 'antiguedad'

In [342]:
df['antiguedad'] = 2026 - df['anio']
df.drop(columns='anio', inplace=True)
df.head(3)

,marca,km,provincia,precio_usd,carroceria,antiguedad
0,Chevrolet,73000,Buenos Aires (A.M.B.A.),6451.61,Hatchback,8
1,Chevrolet,102000,Ciudad Autónoma Buenos Aires,16129.03,Hatchback,3
2,Citroën,84000,Ciudad Autónoma Buenos Aires,10903.23,Minivan,9


In [343]:
# correlacion de las variables numericas

df_corr = df[['km','precio_usd', 'antiguedad']]

matriz_corr = df_corr.corr(method='spearman') # la relación no es lineal pero sí monotónica 
print(matriz_corr)

                  km  precio_usd  antiguedad
km          1.000000   -0.239048    0.708473
precio_usd -0.239048    1.000000   -0.427211
antiguedad  0.708473   -0.427211    1.000000


* km ↔ antigüedad: 0.72 → relación positiva fuerte (a mayor antigüedad, más km).

* precio_usd ↔ antigüedad: -0.43 → relación negativa moderada (autos más viejos tienden a valer menos).

* precio_usd ↔ km: -0.25 → relación negativa débil (más km suele bajar el precio, pero no tan marcado).

# Encoding KNN

KNN mide distancias entre autos. Sin escalar, km (que llega a cientos de miles) aplastaría a las demás columnas en esa distancia. StandardScaler deja todas las columnas en una escala comparable (media 0, desvío 1). fit_transform en train (aprende la escala y la aplica); transform en test (aplica la misma escala aprendida, sin volver a aprenderla).

In [344]:
df_encoded = df.copy()


In [345]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X = df_encoded[['km','marca','provincia', 'carroceria', 'antiguedad']]
y = df_encoded['precio_usd']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [346]:
# 2. Encoding calculado SOLO con y_train
media_provincia = y_train.groupby(X_train['provincia']).mean()
media_carroceria = y_train.groupby(X_train['carroceria']).mean()
media_marca = y_train.groupby(X_train['marca']).mean()

In [347]:
X_train = X_train.copy()
X_test = X_test.copy()

In [348]:
media_global = y_train.mean()
X_train['provincia_enc'] = X_train['provincia'].map(media_provincia)
X_test['provincia_enc']  = X_test['provincia'].map(media_provincia)
X_train['carroceria_enc'] = X_train['carroceria'].map(media_categoria)
X_test['carroceria_enc']  = X_test['carroceria'].map(media_categoria)
X_train['marca_enc'] = X_train['marca'].map(media_marca).fillna(media_global)
X_test['marca_enc']  = X_test['marca'].map(media_marca).fillna(media_global)

In [349]:
X_train

,km,marca,provincia,carroceria,antiguedad,provincia_enc,carroceria_enc,marca_enc
1523,65000,Chevrolet,Córdoba,Hatchback,5,13815.218380,9387.206360,11098.461853
1490,150000,Chevrolet,Buenos Aires,SUV,10,13627.031060,17454.984524,11098.461853
1492,90000,Volkswagen,Ciudad Autónoma Buenos Aires,SUV,4,11585.123734,17454.984524,11248.164226
2780,175000,Volkswagen,Mendoza,Hatchback,20,11744.128125,9387.206360,11248.164226
1908,23000,Ford,Buenos Aires (A.M.B.A.),Pickup,3,13306.518662,19547.940855,13371.772026
...,...,...,...,...,...,...,...,...
4459,156000,Volkswagen,Ciudad Autónoma Buenos Aires,Minivan,16,11585.123734,9546.655101,11248.164226
473,121000,Citroën,Mendoza,SUV,11,11744.128125,17454.984524,9998.325988
3117,90000,Ford,Buenos Aires (A.M.B.A.),Pickup,4,13306.518662,19547.940855,13371.772026
3800,189000,FIAT,Buenos Aires,Furgón,14,13627.031060,13230.319722,9991.715074


In [350]:
# X_train.head()
X_train = X_train.drop(columns=['provincia', 'carroceria','marca'])
X_test  = X_test.drop(columns=['provincia', 'carroceria', 'marca'])

In [351]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

In [352]:
# 3. Escalado (ahora sí, fit solo en train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Baseline KNN

KNN Baseline (SIN MARCA)
----------------
* MAE: 4480.399336944745
* RMSE: 7007.986020245863
* R²: 0.3950399562051433
-----------------
KNN Baseline (CON MARCA ENCODED)
----------------
* MAE: 3370.1328689057423
* RMSE: 5643.469719725168
* R²: 0.6076870572276154

In [353]:
# ¿cuántos NaN hay y en qué columna?
X_test.isna().sum()

km                0
antiguedad        0
provincia_enc     0
carroceria_enc    0
marca_enc         0
dtype: int64

In [354]:
# Baseline valores de metricas sin ajustar ningun hiperparametros

knn = KNeighborsRegressor(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("KNN Baseline")
print("----------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

KNN Baseline
----------------
MAE: 3370.1328689057423
RMSE: 5643.469719725168
R²: 0.6076870572276154


k = cantidad de registros vecinos que se usan para la prediccion

MAE = Error absoluto promedio ¿ cuanto se equivoca el modelo ? 

RMSE = Raiz cuadrada del error elevado al cuadrado. Penaliza los errores mas grandes. 

* Si MAE similar a RMSE, el modelo no tiene valores de error uniformes
* Si MAE << RMSE existen errores grandes

R2 = ¿Qué proporción de la variabilidad de mi variable objetivo consigue explicar el modelo?

In [355]:
import plotly.express as px 

mae = mean_absolute_error(y_test, y_pred)

scatter_base = px.scatter(x = y_test,
                     y = y_pred,
                     labels = {'x':'Precio Real', 'y':'Precio Predicho'},
                     title = 'Valor real versus valor predicho - BASELINE',
                     template = 'plotly_dark')

minimo = min(y_test.min(), y_pred.min())
maximo = max(y_test.max(), y_pred.max())

scatter_base.add_shape(type='line',
                     x0=minimo, y0=minimo,
                     x1=maximo, y1=maximo,
                     line=dict(color='red', dash='dash'))

scatter_base.show()

In [356]:
# Buscar mejor n_neighbors 

resultados_k = []

for k in range(1, 40):

    knn = KNeighborsRegressor(
        n_neighbors = k
    )

    knn.fit(X_train_scaled, y_train)

    y_pred = knn.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados_k.append({
        "k": k,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

import pandas as pd

df_k = pd.DataFrame(resultados_k)

mejor_k_rmse = df_k.loc[df_k["RMSE"].idxmin()]

print("Mejor K según RMSE:")
print(mejor_k_rmse)

Mejor K según RMSE:
k         13.000000
MAE     3274.893121
RMSE    5438.939731
R2         0.635608
Name: 12, dtype: float64


In [357]:
# weights = "distance"
'''
resultados_weights = []

for weight in ["uniform", "distance"]:

    knn = KNeighborsRegressor(
        n_neighbors = 33,   # ya con el valor de k obtenido en la prueba anterior 
        weights = weight    # uniform, 3805.696475, 6077.694781, 0.533536
    )

    knn.fit(X_train_scaled, y_train)

    y_pred = knn.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados_weights.append({
        "weights": weight,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
display(resultados_weights)
'''
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def evaluar_modelo(y_real, y_pred):

    mae = mean_absolute_error(y_real, y_pred)

    mse = mean_squared_error(y_real, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_real, y_pred)

    print(f"MAE:  {mae:.4f}")
    print(f"MSE:  {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")


knn = KNeighborsRegressor(
    n_neighbors = 13,
    weights = "distance",
    p = 2)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

evaluar_modelo(y_test, y_pred)


MAE:  3247.8281
MSE:  30137244.4660
RMSE: 5489.7399
R²:   0.6288


In [358]:
import plotly.express as px 

scatter_p = px.scatter(x = y_test,
                     y = y_pred,
                     labels = {'x':'Precio Real', 'y':'Precio Predicho'},
                     title = 'Valor real versus valor predicho, FEATURES ✅ ',
                     template = 'plotly_dark')

minimo = min(y_test.min(), y_pred.min())
maximo = max(y_test.max(), y_pred.max())

scatter_p.add_shape(type='line',
                     x0=minimo, y0=minimo,
                     x1=maximo, y1=maximo,
                     line=dict(color='red', dash='dash'))

scatter_p.show()

In [303]:
import joblib
import os
print(os.getcwd())

carpeta_destino = 'g:\Mi unidad\Price_pred'

joblib.dump(knn, os.path.join(carpeta_destino,"knn_model.pkl"))
joblib.dump(scaler, os.path.join(carpeta_destino,"scaler.pkl"))

print("Modelo y scaler guardados correctamente.")


g:\Mi unidad\Price_pred\notebooks
Modelo y scaler guardados correctamente.


<>:5: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:5: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\soljo\AppData\Local\Temp\ipykernel_16536\1674369564.py:5: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  carpeta_destino = 'g:\Mi unidad\Price_pred'


In [359]:
import os
import joblib

artefactos = {
    'modelo': knn,
    'scaler': scaler,
    'media_provincia': media_provincia,
    'media_marca': media_marca,
    'media_carroceria': media_carroceria, 
    'columnas': X_train.columns.tolist(),
    'media_global': y_train.mean(),
}

carpeta_destino = r"G:\Mi unidad\Price_pred"
joblib.dump(artefactos, os.path.join(carpeta_destino, 'modelo_predictor.pkl'))

print("Modelo guardado correctamente.")

Modelo guardado correctamente.
